In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

Define Paths

In [3]:
RAW_DATA_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\raw"
PROCESSED_DATA_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\cleaned"

print("Raw Data Path :", RAW_DATA_PATH)
print("Processed Path :", PROCESSED_DATA_PATH)

Raw Data Path : C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\raw
Processed Path : C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\cleaned


Load Data

In [4]:
sales = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_sales.csv")

inventory = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_inventory.csv")

sku = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_skus.csv")

stores = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_stores.csv")

customers = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_customers.csv")

promotions = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_promotions.csv")

Data Shape

In [5]:
datasets = {
    "Sales": sales,
    "Inventory": inventory,
    "SKU": sku,
    "Stores": stores,
    "Customers": customers,
    "Promotions": promotions
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Sales: (641843, 9)
Inventory: (8735, 7)
SKU: (200, 7)
Stores: (50, 5)
Customers: (5000, 7)
Promotions: (33, 6)


Rename Customer key

In [6]:
if "cust_id" in customers.columns:
    customers.rename(columns={"cust_id": "customer_id"}, inplace=True)

print(customers.columns)

Index(['customer_id', 'age', 'gender', 'city', 'loyalty_segment',
       'preferred_channel', 'registration_date'],
      dtype='object')


Remove Duplicate

In [7]:
for name, df in datasets.items():

    before = df.shape[0]

    df.drop_duplicates(inplace=True)

    after = df.shape[0]

    print(f"{name}")

    print("Removed:", before-after)

Sales
Removed: 45
Inventory
Removed: 0
SKU
Removed: 0
Stores
Removed: 0
Customers
Removed: 0
Promotions
Removed: 0


Missing Value Report

In [8]:
for name, df in datasets.items():

    print("="*50)

    print(name)

    print(df.isnull().sum())

Sales
date                 0
store_id             0
sku_id               0
customer_id     159777
quantity             0
unit_price           0
total_value          0
channel              0
discount_pct         0
dtype: int64
Inventory
store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
snapshot_date        0
dtype: int64
SKU
sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64
Stores
store_id        0
store_name      0
city            0
store_type      0
opening_date    0
dtype: int64
Customers
customer_id          0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64
Promotions
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
promo_id        0
dtype: int64


Fill Numeric Value

In [9]:
for name, df in datasets.items():

    numeric_columns = df.select_dtypes(include=np.number).columns

    for col in numeric_columns:

        df[col].fillna(df[col].median(), inplace=True)

C:\Users\mayank\AppData\Local\Temp\ipykernel_19684\3740879478.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


Fill Categorical Value

In [10]:
for name, df in datasets.items():

    categorical_columns = df.select_dtypes(include="object").columns

    for col in categorical_columns:

        df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\mayank\AppData\Local\Temp\ipykernel_19684\352862384.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
C:\Users\mayank\AppData\Local\Temp\ipykernel_19684\352862384.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

Convert Date column

In [11]:
for name, df in datasets.items():

    for col in df.columns:

        if "date" in col.lower():

            df[col] = pd.to_datetime(df[col], errors="coerce")

Standardize Text Columns

In [12]:
for name, df in datasets.items():

    text_columns = df.select_dtypes(include="object").columns

    for col in text_columns:

        df[col] = df[col].str.strip()

        df[col] = df[col].str.title()

Remove Negative Quantity

In [13]:
if "quantity" in sales.columns:

    sales = sales[sales["quantity"] >= 0]

Remove Negative Sales Amount

In [14]:
if "sales_amount" in sales.columns:

    sales = sales[sales["sales_amount"] >= 0]

Remove Negative Inventory

In [15]:
if "stock_quantity" in inventory.columns:

    inventory = inventory[inventory["stock_quantity"] >= 0]

Outlier Detection (IQR)

In [16]:
def remove_outliers(df):

    numeric = df.select_dtypes(include=np.number).columns

    for col in numeric:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR

        upper = Q3 + 1.5 * IQR

        df = df[(df[col] >= lower) & (df[col] <= upper)]

    return df

Apply Outlier Removal

In [17]:
sales = remove_outliers(sales)

inventory = remove_outliers(inventory)

Merge SKU Information

In [18]:
if "sku_id" in sales.columns and "sku_id" in sku.columns:

    sales = sales.merge(

        sku,

        on="sku_id",

        how="left"
    )

Merge Store Information

In [19]:
if "store_id" in sales.columns and "store_id" in stores.columns:

    sales = sales.merge(

        stores,

        on="store_id",

        how="left"
    )

Merge Customer Information

In [20]:
if "customer_id" in sales.columns and "customer_id" in customers.columns:

    sales = sales.merge(

        customers,

        on="customer_id",

        how="left"
    )

Merge Promotions

In [21]:
if "promotion_id" in sales.columns and "promotion_id" in promotions.columns:

    sales = sales.merge(

        promotions,

        on="promotion_id",

        how="left"
    )

Merge Inventory

In [23]:
if "sku_id" in sales.columns and "sku_id" in inventory.columns:

    sales = sales.merge(

        inventory,

        on="sku_id",

        how="left",

        suffixes=("", "_inventory")
    )

MemoryError: Unable to allocate 165. MiB for an array with shape (21672402,) and data type int64

Final Dataset Information

In [ ]:
print(sales.shape)

sales.head()

Remaining Missing Values

In [ ]:
sales.isnull().sum().sort_values(ascending=False)

Fill Remaining Missing Values

In [ ]:
sales.fillna(method="ffill", inplace=True)

sales.fillna(method="bfill", inplace=True)

Final Duplicate Check

In [ ]:
print("Duplicates:", sales.duplicated().sum())

: 

Save Cleaned Sales

In [ ]:
sales.to_csv(

    PROCESSED_DATA_PATH + "cleaned_sales.csv",

    index=False
)

print("cleaned_sales.csv Saved")

Save Cleaned Inventory

In [ ]:
inventory.to_csv(

    PROCESSED_DATA_PATH + "\\" + "cleaned_inventory.csv",

    index=False
)

print("cleaned_inventory.csv Saved")

Save Final Dataset

In [ ]:
sales.to_csv(

    PROCESSED_DATA_PATH + "final_dataset.csv",

    index=False
)

print("final_dataset.csv Saved")

Cleaning Summary

In [ ]:
summary = pd.DataFrame({

    "Dataset": [
        "Sales",
        "Inventory",
        "Final Dataset"
    ],

    "Rows": [
        sales.shape[0],
        inventory.shape[0],
        sales.shape[0]
    ],

    "Columns": [
        sales.shape[1],
        inventory.shape[1],
        sales.shape[1]
    ]
})

summary